# Add data via SparQL queries

## 1 Import needed packages

In [ ]:
import requests
import pandas as pd
from tqdm.notebook import tqdm
import numpy as np
from constants import DATA_PATH, TMP_PATH
from typing import Optional
from utils import check_and_create_file

tqdm.pandas()

## 2 Add data from dbpedia

### 2.1 Query, Endpoint and Parameters

In [2]:
# Your SPARQL query
sparql_query = """
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
PREFIX dbc: <http://dbpedia.org/resource/Category:>

SELECT DISTINCT ?entry ?entryLabel ?form ?number ?found
WHERE {
	VALUES ?concept {
		dbc:Greek_New_Testament_lectionaries
		dbc:Greek_New_Testament_minuscules
		dbc:Greek_New_Testament_uncials
		dbc:New_Testament_papyri
	}
	?entry dcterms:subject ?concept .
	
	OPTIONAL{?entry rdfs:label ?entryLabel}
	OPTIONAL{?entry dbp:form ?form}
	OPTIONAL{?entry dbp:number ?number}
	OPTIONAL{?entry dbp:found ?found}
	
	FILTER (langMatches(lang(?entryLabel), "en"))
}
"""

# DBpedia SPARQL endpoint
sparql_endpoint = "https://dbpedia.org/sparql"


# Set the request parameters
params = {"query": sparql_query, "format": "json"}

### 2.2 Request data

In [3]:
# Send the SPARQL query to DBpedia
response = requests.get(sparql_endpoint, params=params)

# Check if the request was successful
if response.status_code == 200:
    # Parse the JSON response
    data = response.json()

    # Extract the bindings from the response
    bindings = data["results"]["bindings"]

    # Convert the bindings to a list of dictionaries
    results_list = [
        {key: binding[key]["value"] for key in binding} for binding in bindings
    ]

    # Create a pandas DataFrame from the list
    manuscripts_sparql_df = pd.DataFrame(results_list)

else:
    print(f"Error: {response.status_code} - {response.text}")

### 2.3 Cleanup 'number' column

In [ ]:
def has_decimal(string: str) -> bool:
    value = float(string)
    return value % 1 != 0


# Custom function to clean and convert values to integers
def clean_and_convert(string: str) -> Optional[int]:
    try:
        if has_decimal(string):
            return None
        else:
            cleaned_value = "".join(filter(str.isdigit, string))
            return int(cleaned_value) if cleaned_value else None
    except:
        return None


manuscripts_cleanup1_df = manuscripts_sparql_df.copy()

# Apply the custom function to the specified column
manuscripts_cleanup1_df["number"] = manuscripts_cleanup1_df["number"].progress_apply(
    clean_and_convert
)
# if a number is greater than 3000 (by mistake) set it to None
manuscripts_cleanup1_df.loc[manuscripts_cleanup1_df["number"] > 3000, "number"] = None

### 2.4 Cleanup 'found' column

In [5]:
manuscripts_cleanup2_df = manuscripts_cleanup1_df.copy()

# Fill NaN values in the 'found' column with an empty string
manuscripts_cleanup2_df["found"] = (
    manuscripts_cleanup2_df["found"].fillna("").astype(str)
)
# run a groupby to merge found entries of otherwise duplicate rows
manuscripts_cleanup2_df = manuscripts_cleanup2_df.groupby(
    ["entry", "entryLabel", "form", "number"], as_index=False
)["found"].agg(",".join)

### 2.5 (Re)Generate the GA number from manuscript 'number' and 'form'

In [ ]:
# Custom function to modify values based on the 'form' column
def generate_ga(row):
    if pd.notna(row["form"]) and pd.notna(row["number"]):
        if row["form"] == "Papyrus":
            return "P" + str(int(row["number"]))
        elif row["form"] == "Uncial":
            return "0" + str(int(row["number"]))
        elif row["form"] == "Minuscule":
            return str(int(row["number"]))
        elif row["form"] == "Lectionary":
            return "L" + str(int(row["number"]))
    else:
        return None


manuscripts_cleanup3_df = manuscripts_cleanup2_df.copy()

manuscripts_cleanup3_df["ga"] = manuscripts_cleanup3_df.progress_apply(
    generate_ga, axis=1
)

manuscripts_cleanup3_df = manuscripts_cleanup3_df.rename(
    columns={"entry": "dbpedia", "entryLabel": "label"}
)
manuscripts_cleanup3_df.drop(labels=["number", "form"], axis=1, inplace=True)
manuscripts_cleanup3_df["source"] = "dbpedia"
manuscripts_cleanup3_df["dbpedia"] = manuscripts_cleanup3_df["dbpedia"].str.replace(
    "http://dbpedia.org/resource/", "", regex=False
)

### 2.6 Merge with already known data

In [ ]:
# read file manuscripts.csv
manuscripts_df = pd.read_csv(
    TMP_PATH + "manuscripts_json_tei.csv",
    low_memory=False,
    dtype={
        "docID": "string",
        "pagesCount": "Int64",
        "leavesCount": "Int64",
        "ga": "string",
        "century": "string",
        "source": "string",
        "label": "string",
    },
)

In [8]:
merged_df = pd.concat([manuscripts_df, manuscripts_cleanup3_df], ignore_index=True)

### 2.7 Cleanup

In [9]:
# drop unnecessary column
merged_df = merged_df.drop(columns=["found"])

In [ ]:
# Define a function to aggregate rows
def merge_rows(group):
    merged_row = {"ga": group["ga"].iloc[0], "source": group["source"].iloc[0]}

    # Iterate over other columns and create a set of non-null/non-NaN values
    for col in group.columns:
        if col not in ["ga", "source"]:
            merged_values = set(group[col])
            # Select the first non-null/non-NaN value
            merged_row[col] = next(
                (value for value in merged_values if pd.notna(value)), np.nan
            )

    return pd.Series(merged_row)


# Group by 'ga' and 'source', then apply the merge_rows function
merged_df = merged_df.groupby(["ga", "source"]).apply(merge_rows).reset_index(drop=True)

merged_df.head(-1)

In [ ]:
# set None Types
merged_df["docID"] = merged_df["docID"].fillna("NA").astype(str)
merged_df["pagesCount"] = merged_df["pagesCount"].fillna(-1).astype(int)
merged_df["leavesCount"] = merged_df["leavesCount"].fillna(-1).astype(int)
merged_df["ga"] = merged_df["ga"].fillna("NA").astype(str)
merged_df["century"] = merged_df["century"].fillna("NA").astype(str)
merged_df["source"] = merged_df["source"].fillna("NA").astype(str)
merged_df["label"] = merged_df["label"].fillna("NA").astype(str)
merged_df["dbpedia"] = merged_df["dbpedia"].fillna("NA").astype(str)
merged_df["label"] = merged_df["label"].fillna("NA").astype(str)

column_types = {
    "docID": "string",
    "pagesCount": "Int64",
    "leavesCount": "Int64",
    "ga": "string",
    "century": "string",
    "source": "string",
    "dbpedia": "string",
    "label": "string",
}
merged_df.astype(column_types)

## 3 Writing to file

In [ ]:
check_and_create_file(TMP_PATH + "manuscripts.csv")
merged_df.to_csv(TMP_PATH + "manuscripts.csv", index=False)